[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ncsu-geoforall-lab/gis-584-uas-course/blob/gh-pages/course/topics/topic_6_change_detection/assignments/vegetation_indices.ipynb)

# Lab: Vegetation indices from UAS imagery

**Course:** GIS/MAE 584 Mapping and Analytics Using UAS
**Institution:** NC State University, Center for Geospatial Analytics
**Instructor:** Corey White

Spectral indices turn the bands of an orthomosaic into one number per cell that tracks a property of the surface, mostly the amount of green vegetation. They are the first features of the image classification in Assignment 6A. In this notebook we compute indices from two kinds of UAS imagery: the RGB orthophoto of Lake Wheeler from July 2024, where only visible bands are available, and a 10-band MicaSense RedEdge-MX Dual orthomosaic from a USGS calibration campaign, where near infrared and red edge bands allow the standard vegetation indices. The USGS data come in two calibrations of the same flight, so we also see how much the calibration changes an index.

This notebook runs the same steps as the [lab page](https://ncsu-geoforall-lab.github.io/gis-584-uas-course/course/topics/topic_6_change_detection/assignments/lab_vegetation_indices.html) in Python. You can run it in Google Colab (badge above) or locally with GRASS 8.5 installed.

## Learning objectives

- Separate the bands of an orthomosaic and compute indices with map algebra
- Compare RGB indices (VARI, GLI, ExG) with NIR based indices (NDVI, NDRE, SAVI, GNDVI)
- Read index values at known surfaces and set a threshold for green vegetation
- Explain why reflectance calibration matters for indices

## Outline

- Part 1: Environment setup
- Part 2: RGB indices, Lake Wheeler 2024
- Part 3: NIR and red edge indices, USGS EROS 2023
- Part 4: Calibration and the index
- Report questions

---
## Part 1: Environment setup

### Install GRASS 8.5

This notebook uses the `grass.tools` Python API introduced in GRASS 8.5, so we install GRASS 8.5 from the conda-forge channel using [condacolab](https://github.com/conda-incubator/condacolab).

> **Important: the next cell restarts the Colab runtime.** After you run it, Colab will report "Your session crashed for an unknown reason". This is expected and happens exactly once. Wait a few seconds for the runtime to reconnect, then continue with the next cell. If you use "Run all", run it again after the restart: the install cell detects the existing conda installation and returns immediately.

If you are running locally with GRASS 8.5 already installed, skip the three install cells.

In [ ]:
!pip install -q condacolab
import condacolab

condacolab.install()

After the runtime restarts, verify that conda is ready.

In [ ]:
import condacolab

condacolab.check()

Install GRASS 8.5 from conda-forge with mamba. This takes a few minutes.

In [ ]:
!mamba install -y -q grass=8.5

Check that GRASS is installed and which Python version is running.

In [ ]:
!grass --version

import sys

v = sys.version_info
print(f"We are using Python {v.major}.{v.minor}.{v.micro}")

### Import the GRASS Python packages

GRASS tells us where its Python packages live with `grass --config python_path`. We add that to `sys.path`, then import `grass.script` (the classic API), `grass.jupyter` (maps in notebooks), and `grass.tools` (every GRASS tool as a Python function, new in 8.5). We also import pandas and matplotlib for the tables and plots.

In [ ]:
import io
import os
import subprocess
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Ask GRASS where its Python packages are.
sys.path.append(
    subprocess.check_output(["grass", "--config", "python_path"], text=True).strip()
)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools

### Download the data

- The `Lake_Wheeler_NCspm` GRASS project (EPSG:3358, NC State Plane meters, about 225 MB)

The orthomosaics are cloud optimized GeoTIFFs on the course bucket; `r.import` reads only the part of them we ask for, so they are not downloaded as files.

In [ ]:
DATA_DIR = Path("/content")  # change this if you run locally
DATA_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_DIR)

BUCKET = "https://storage.googleapis.com/gis-course-data/gis584"
PROJECT_URL = f"{BUCKET}/grassdata/Lake_Wheeler_NCspm.zip"
ORTHO_2024_URL = f"{BUCKET}/uas-flight-data/Lake%20Wheeler%20-%20NCSU/071724/odm_orthophoto.tif"
EROS_PANELS_URL = f"{BUCKET}/uas-flight-data/EROS_ECCOE_USGS/071823/2023-07-18_1200_MicaSenseDual_ortho_panels_cog.tif"
EROS_ELM_URL = f"{BUCKET}/uas-flight-data/EROS_ECCOE_USGS/071823/2023-07-18_1200_MicaSenseDual_ortho_panels_Reflectance_ELM_cog.tif"

!curl -sSL -o Lake_Wheeler_NCspm.zip {PROJECT_URL} && unzip -q -o Lake_Wheeler_NCspm.zip
!ls -lh {DATA_DIR}

### Start the GRASS session

We create a new mapset `indices` inside the Lake Wheeler project for Part 2, start the session there, and create a `Tools` object. Part 3 switches to a second project in the coordinate system of the USGS site.

Colab presets PROJ and GDAL to an older system install, which breaks reprojection with the conda GRASS, so we point them at the conda data files first.

In [ ]:
# Point PROJ and GDAL at the conda data files. Colab presets PROJ_LIB and
# GDAL_DATA to the older system install in /usr/share, which crashes the
# newer conda PROJ, so these must be overridden, not just defaulted.
proj_data = Path(sys.prefix) / "local" / "share" / "proj"
gdal_data = Path(sys.prefix) / "local" / "share" / "gdal"
if proj_data.is_dir():
    os.environ["PROJ_DATA"] = str(proj_data)
    os.environ["PROJ_LIB"] = str(proj_data)
if gdal_data.is_dir():
    os.environ["GDAL_DATA"] = str(gdal_data)

project = DATA_DIR / "Lake_Wheeler_NCspm"
mapset = project / "indices"

# Create the mapset once; -c creates, -e exits without starting a shell.
if not mapset.is_dir():
    subprocess.run(["grass", "-c", "-e", str(mapset)], check=True)

session = gj.init(mapset)
tools = Tools(session=session, overwrite=True)

print(tools.g_proj(flags="p").text)

### Helpers

`describe` prints the statistics of a raster in the current region. `show` draws a raster with a legend. `rgb` draws a three band composite. `areas` returns the area of each category of a raster as a table.

In [ ]:
def describe(raster, label=None):
    """Print the range, mean, and spread of a raster in the current region."""
    s = tools.r_univar(map=raster, flags="e", format="json").json
    print(f"{label or raster}: {s['n']:,} cells")
    print(f"  min {s['min']:8.3f}  max {s['max']:8.3f}  mean {s['mean']:8.3f}  median {s['median']:8.3f}  stddev {s['stddev']:6.3f}")


def show(raster, title=None, legend=None, width=500, height=400, flags="sb", vector=None, range=None):
    """Draw one raster in the current region with a legend."""
    m = gj.Map(width=width, height=height, use_region=True)
    m.d_rast(map=raster)
    if vector:
        m.d_vect(map=vector, icon="basic/circle", size=12, fill_color="yellow", color="black",
                 attribute_column="label", label_color="yellow", label_size=12, xref="left")
    if legend and "n" in flags:
        m.d_legend(raster=raster, at=(4, 24, 3, 7), flags=flags + "b", fontsize=11, bgcolor="white")
    elif legend:
        m.d_legend(raster=raster, at=(5, 45, 3, 6), title=legend, flags=flags, bgcolor="white", range=range)
    m.d_barscale(at=(55, 5))
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()
    return m


def rgb(red, green, blue, title=None, width=500, height=400, vector=None):
    """Draw a true color composite of three bands."""
    m = gj.Map(width=width, height=height, use_region=True)
    m.d_rgb(red=red, green=green, blue=blue)
    if vector:
        m.d_vect(map=vector, icon="basic/circle", size=12, fill_color="yellow", color="black",
                 attribute_column="label", label_color="yellow", label_size=12, xref="left")
    m.d_barscale(at=(55, 5))
    if title:
        m.d_text(text=title, at=(3, 94), size=4, color="black", bgcolor="white")
    m.show()
    return m


def areas(raster):
    """Area in hectares and percent of each category of a raster in the current region."""
    text = tools.r_stats(input=raster, flags="anl", separator="|").text
    table = pd.read_csv(io.StringIO(text), sep="|", names=["category", "label", "area_m2"])
    table["hectares"] = table["area_m2"] / 10000
    table["percent"] = 100 * table["area_m2"] / table["area_m2"].sum()
    return table[["category", "label", "hectares", "percent"]].round(2)

---
## Part 2: RGB indices, Lake Wheeler 2024

### Import a block of the orthophoto

The July 2024 orthophoto of Lake Wheeler was processed in WebODM and is stored as a cloud optimized GeoTIFF in UTM zone 17N with four bands: red, green, blue, and an alpha band that marks the area outside the mosaic. We set the region to a 500 by 400 m block of the fields at 0.2 m resolution and let `r.import` read, reproject, and resample only that block (`extent=region`, `resolution=region`). The original resolution is 7 cm; 20 cm is enough for indices and ten times faster.

In [ ]:
tools.g_region(n=219700, s=219300, w=636700, e=637200, res=0.2, flags="p")
tools.r_import(input=f"/vsicurl/{ORTHO_2024_URL}", output="ortho_2024", extent="region", resolution="region")
print(tools.g_list(type="raster", pattern="ortho_2024*", mapset=".").text)
describe("ortho_2024.1", "red band")

#### Map: the block in true color

Five sample points mark surfaces we will read the indices at: bare soil, crop rows, tree canopy, the paved road, and a grassed waterway.

In [ ]:
points = """636733|219551|bare soil
636853|219634|crop
637135|219480|trees
637150|219625|paved road
637050|219490|grass
"""
tools.v_in_ascii(input=io.StringIO(points), output="samples", separator="pipe",
                 columns="x double precision, y double precision, label text")
rgb("ortho_2024.1", "ortho_2024.2", "ortho_2024.3", title="Lake Wheeler, July 17, 2024", vector="samples", width=625, height=500)

### Indices from the visible bands

Without a near infrared band, the indices use the contrast between green and red, which chlorophyll creates in the visible range. The bands are 8-bit digital numbers, so the expressions use `float()` to avoid integer division:

- VARI, visible atmospherically resistant index: (G - R) / (G + R - B). The denominator is near zero on very dark cells, so the expression sets zero to null and clamps the result to -1 to 1.
- GLI, green leaf index: (2G - R - B) / (2G + R + B)
- ExG, excess green: 2g - r - b, where g, r, b are the band values divided by their sum
- BI, brightness: the root mean square of the three bands; not a vegetation index, but it separates bright soil and pavement from everything else

The `ndvi` color table runs from red through white to green between -1 and 1, which suits any normalized index.

In [ ]:
R, G, B = "ortho_2024.1", "ortho_2024.2", "ortho_2024.3"
tools.r_mapcalc(expression=f"vari = if({G} + {R} - {B} == 0, null(), max(-1.0, min(1.0, float({G} - {R}) / ({G} + {R} - {B}))))")
tools.r_mapcalc(expression=f"gli = float(2 * {G} - {R} - {B}) / (2 * {G} + {R} + {B})")
tools.r_mapcalc(expression=f"exg = (2.0 * {G} - {R} - {B}) / ({R} + {G} + {B})")
tools.r_mapcalc(expression=f"bi = sqrt(({R} * {R} + {G} * {G} + {B} * {B}) / 3.0)")
for index in ["vari", "gli", "exg"]:
    tools.r_colors(map=index, color="ndvi")
tools.r_colors(map="bi", color="grey", flags="e")
for index in ["vari", "gli", "exg", "bi"]:
    describe(index)

In [ ]:
show("vari", title="VARI", legend="VARI", width=625, height=500, vector="samples")
show("gli", title="GLI", legend="GLI", width=625, height=500)

### Index values at the sample points

`r.what` reads every raster at the sample points. VARI and GLI are close to zero on soil and pavement and rise with the amount of green leaf; the crop point falls between because the 20 cm cells mix leaves and the soil between rows.

In [ ]:
text = tools.r_what(map=[R, G, B, "vari", "gli", "exg", "bi"], points="samples", separator="|").text
values = pd.read_csv(io.StringIO(text), sep="|", names=["x", "y", "site", "red", "green", "blue", "vari", "gli", "exg", "bi"])
values["label"] = ["bare soil", "crop", "trees", "paved road", "grass"]
values.set_index("label")[["red", "green", "blue", "vari", "gli", "exg", "bi"]].round(3)

### A threshold map

A simple classification by VARI: below 0 is bare soil and pavement, 0 to 0.1 is mixed, above 0.1 is green vegetation. `r.category` labels the classes and `r.stats -a` gives their areas.

In [ ]:
tools.r_mapcalc(expression="vari_class = if(vari < 0, 1, if(vari < 0.1, 2, 3))")
tools.r_category(map="vari_class", separator=":", rules=io.StringIO("1:bare and built\n2:mixed\n3:green vegetation\n"))
tools.r_colors(map="vari_class", rules=io.StringIO("1 230:200:160\n2 200:220:150\n3 40:140:40\n"))
show("vari_class", title="Classes from VARI", legend="class", flags="n", width=625, height=500)
areas("vari_class")

---
## Part 3: NIR and red edge indices, USGS EROS 2023

### The data

On July 18, 2023 the USGS flew a MicaSense RedEdge-MX Dual camera at 62 m over the calibration site at the EROS Center in South Dakota, with reflectance panels and calibration targets laid out on the ground (Irwin et al. 2025, public domain). The camera has ten bands; the orthomosaic stores them in order of wavelength:

| Band | Center (nm) | Name |
|---|---|---|
| 1 | 444 | coastal blue |
| 2 | 475 | blue |
| 3 | 531 | green (531) |
| 4 | 560 | green |
| 5 | 650 | red (650) |
| 6 | 668 | red |
| 7 | 705 | red edge (705) |
| 8 | 717 | red edge |
| 9 | 740 | red edge (740) |
| 10 | 842 | near infrared |

The values are reflectance, 0 to 1. Two versions of the orthomosaic exist: one calibrated with the manufacturer's panel and sun sensor workflow (`panels`) and one corrected with the empirical line method (`elm`). We use the ELM version here and compare the two in Part 4.

### A second project

The site is in UTM zone 14N, so we create a second GRASS project for it and switch the session there.

In [ ]:
eros_project = DATA_DIR / "eros_eccoe"
if not eros_project.is_dir():
    subprocess.run(["grass", "-c", "EPSG:6343", "-e", str(eros_project)], check=True)
session = gj.init(eros_project / "PERMANENT")
tools = Tools(session=session, overwrite=True)
print(tools.g_proj(flags="p").text)

### Import the two orthomosaics

The region covers 220 by 240 m of the site with the grass field, crop rows, a gravel road, the asphalt road with the calibration targets, and trees, at 10 cm (the orthomosaics are 4.2 cm). Each import creates ten rasters, one per band, and takes a minute or two because the bands are read from the cloud optimized GeoTIFF on the bucket.

In [ ]:
tools.g_region(n=4845640, s=4845400, w=690700, e=690920, res=0.1, flags="pa")
tools.r_import(input=f"/vsicurl/{EROS_ELM_URL}", output="elm_raw", extent="region", resolution="region")
tools.r_import(input=f"/vsicurl/{EROS_PANELS_URL}", output="panels_raw", extent="region", resolution="region")
print(tools.g_list(type="raster", pattern="elm_raw*", mapset=".").text)
describe("elm_raw.10", "NIR reflectance, as delivered")

### Clamp the bands to the physical range

The empirical line fit extrapolates: deep shadow comes out slightly below zero and the white tarp above one. A ratio of two bands near zero gives huge values, so we clamp every band to 0 to 1 before computing indices. The same is done for the manufacturer product so that the two stay comparable.

In [ ]:
for product in ["elm", "panels"]:
    for i in range(1, 11):
        tools.r_mapcalc(expression=f"{product}.{i} = max(0.0, min(1.0, {product}_raw.{i}))")
tools.g_remove(type="raster", pattern="*_raw.*", flags="f");
describe("elm.10", "NIR reflectance, clamped")

In [ ]:
BANDS = {"blue": "elm.2", "green": "elm.4", "red": "elm.6", "rededge": "elm.8", "nir": "elm.10"}
# reflectance is 0 to 1 and the surfaces are dark, so stretch the color tables before drawing
tools.i_colors_enhance(red=BANDS["red"], green=BANDS["green"], blue=BANDS["blue"], strength=95)
rgb(BANDS["red"], BANDS["green"], BANDS["blue"], title="EROS site, July 18, 2023 (ELM reflectance)", width=625, height=500)

### NDVI two ways

`i.vi` computes the common indices from the band rasters; `r.mapcalc` does the same from the formula. We compute NDVI both ways and check that they agree.

In [ ]:
tools.i_vi(output="ndvi", viname="ndvi", red=BANDS["red"], nir=BANDS["nir"])
tools.r_mapcalc(expression=f"ndvi_calc = ({BANDS['nir']} - {BANDS['red']}) / ({BANDS['nir']} + {BANDS['red']})")
tools.r_mapcalc(expression="ndvi_check = abs(ndvi - ndvi_calc)")
describe("ndvi")
describe("ndvi_check", "difference between i.vi and r.mapcalc")

### Red edge, soil adjusted, and green indices

- NDRE: (NIR - RedEdge) / (NIR + RedEdge), with the 717 nm band
- SAVI: (1 + L) (NIR - Red) / (NIR + Red + L) with L = 0.5
- GNDVI: (NIR - Green) / (NIR + Green)

In [ ]:
nir, red, rededge, green = BANDS["nir"], BANDS["red"], BANDS["rededge"], BANDS["green"]
tools.r_mapcalc(expression=f"ndre = ({nir} - {rededge}) / ({nir} + {rededge})")
tools.r_mapcalc(expression=f"savi = 1.5 * ({nir} - {red}) / ({nir} + {red} + 0.5)")
tools.r_mapcalc(expression=f"gndvi = ({nir} - {green}) / ({nir} + {green})")
for index in ["ndvi", "ndre", "savi", "gndvi"]:
    tools.r_colors(map=index, color="ndvi")
    describe(index)

In [ ]:
show("ndvi", title="NDVI", legend="NDVI", width=625, height=500)
show("ndre", title="NDRE", legend="NDRE", width=625, height=500)

#### Plot: the distribution of NDVI

`r.stats -c` counts the cells in equal intervals of the index. A histogram with two groups of values, one for vegetation and one for everything else, suggests where a threshold goes.

In [ ]:
text = tools.r_stats(input="ndvi", flags="cn", nsteps=50, separator="|").text
hist = pd.read_csv(io.StringIO(text), sep="|", names=["range", "count"])
hist["low"] = hist["range"].str.extract(r"^(-?[\d.]+)")[0].astype(float)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(hist["low"], hist["count"], width=hist["low"].diff().median(), align="edge")
ax.set_xlabel("NDVI")
ax.set_ylabel("cells")
plt.tight_layout()
plt.show()

### A threshold map

The histogram has almost no cells between 0.2 and 0.3, so that gap is where the threshold goes. Cells with NDVI above 0.3 are taken as green vegetation; `r.stats -a` gives the area.

In [ ]:
tools.r_mapcalc(expression="green = if(ndvi > 0.3, 1, 2)")
tools.r_category(map="green", separator=":", rules=io.StringIO("1:green vegetation\n2:other\n"))
tools.r_colors(map="green", rules=io.StringIO("1 40:140:40\n2 230:200:160\n"))
show("green", title="NDVI above 0.3", legend="class", flags="n", width=625, height=500)
areas("green")

---
## Part 4: Calibration and the index

The `panels` orthomosaic is the same flight processed with the manufacturer's workflow: the reflectance panel imaged before the flight and the sun sensor on the aircraft. The `elm` orthomosaic was corrected afterwards with the empirical line method, a per-band linear fit between the imagery and reflectance measured on several targets in the field. Shrestha et al. (2025) found the manufacturer workflow about 0.015 too low in reflectance and the ELM within 0.005.

### The same NDVI from both products

In [ ]:
PANELS = {name: band.replace("elm", "panels") for name, band in BANDS.items()}
tools.r_mapcalc(expression=f"ndvi_panels = ({PANELS['nir']} - {PANELS['red']}) / ({PANELS['nir']} + {PANELS['red']})")
tools.r_mapcalc(expression="ndvi_diff = ndvi - ndvi_panels")
tools.r_colors(map="ndvi_diff", rules=io.StringIO("-0.1 blue\n0 white\n0.1 red\n"))  # fixed scale, so a few outliers do not stretch it
describe("ndvi_panels", "NDVI, manufacturer calibration")
describe("ndvi", "NDVI, empirical line method")
describe("ndvi_diff", "ELM minus manufacturer")
show("ndvi_diff", title="NDVI difference, ELM minus manufacturer", legend="difference", range=(-0.1, 0.1), width=625, height=500)

### Reflectance of one target in both products

`r.what` reads all ten bands at one calibration target. The two spectra show the offset the ELM removes; a target of uniform grey should have nearly the same reflectance in every band.

In [ ]:
target = (690865.4, 4845586.9)
wavelengths = [444, 475, 531, 560, 650, 668, 705, 717, 740, 842]
spectra = {}
for product in ["elm", "panels"]:
    rec = tools.r_what(map=[f"{product}.{i}" for i in range(1, 11)], coordinates=target, format="json").json[0]
    spectra[product] = [rec[f"{product}.{i}"]["value"] for i in range(1, 11)]
fig, ax = plt.subplots(figsize=(7, 3.5))
for product, label in [("panels", "manufacturer workflow"), ("elm", "empirical line method")]:
    ax.plot(wavelengths, spectra[product], "o-", label=label)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.legend()
plt.tight_layout()
plt.show()
pd.DataFrame(spectra, index=wavelengths).round(4)

---
## Report questions

Answer in the cells below, or in your report document. Keep the answers short and refer to the outputs above.

#### Question 1: RGB indices

Report the VARI and GLI values at the five sample points and the areas of the three VARI classes. Which surfaces do the RGB indices separate well, and where do they fail?

`Add text here with your answer to question 1`

#### Question 2: NIR and red edge indices

Report the statistics of NDVI, NDRE, SAVI, and GNDVI, the histogram, and the area above the NDVI threshold. Where do NDVI and NDRE differ on the maps, and why?

`Add text here with your answer to question 2`

#### Question 3: Calibration

Report the mean difference between the two NDVI maps and the two spectra of the target. Is the difference large enough to matter for the threshold map? When would it matter?

`Add text here with your answer to question 3`

#### Question 4: Choosing an index

For the Lake Wheeler classification in Assignment 6A, which index would you use and what would you gain with a multispectral camera?

`Add text here with your answer to question 4`